# Depth Model Comparison for Corn Seed Images

Benchmark 6 monocular depth estimation models on 200 random corn seed images.
Evaluates: edge alignment, seed/background contrast, intra-seed smoothness, and speed.

**GPU:** A100 recommended. T4 (free) also works — all models fit in 16GB.

**Models:** Depth Pro (Apple), MoGe-2 (Microsoft), DepthFM (CompVis), Pixel-Perfect Depth, VGGT (Meta), Depth Anything V3 (ByteDance)

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b colab https://github.com/shurjo05/DepthComparison.git /content/DepthComparison 2>/dev/null || echo "Already cloned"
%cd /content/DepthComparison

## 2. Set Dataset Path

**Edit the path below** to point to your corn seed dataset on Google Drive.  
The folder should contain `images/` and `labels/` subdirectories.

In [ ]:
import os

# >>> EDIT THIS PATH to match your Google Drive folder <<<
os.environ["DATASET_DIR"] = "/content/drive/MyDrive/CornSeedDetection6D/test/test"

# Verify the dataset exists
img_dir = os.path.join(os.environ["DATASET_DIR"], "images")
lbl_dir = os.path.join(os.environ["DATASET_DIR"], "labels")
assert os.path.isdir(img_dir), f"Images dir not found: {img_dir}"
assert os.path.isdir(lbl_dir), f"Labels dir not found: {lbl_dir}"
n_images = len([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
print(f"✓ Found {n_images} images in {img_dir}")

## 3. Install Dependencies & Clone Model Repos

Installs pip deps and clones repos that require local checkouts. ~3-5 min.

In [ ]:
!pip install -q -r requirements.txt

# MoGe-2 (Microsoft) — pip installable
!pip install -q git+https://github.com/microsoft/MoGe.git

# Pixel-Perfect Depth
!git clone https://github.com/gangweix/pixel-perfect-depth 2>/dev/null || echo "already cloned"
!cd pixel-perfect-depth && pip install -q -r requirements.txt

# VGGT (Meta)
!git clone https://github.com/facebookresearch/vggt 2>/dev/null || echo "already cloned"
!cd vggt && pip install -q -r requirements.txt

# Depth Anything V3 (ByteDance)
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3 2>/dev/null || echo "already cloned"
!cd Depth-Anything-3 && pip install -q -e .

# DepthFM (CompVis) — repo + checkpoint download
!git clone https://github.com/CompVis/depth-fm 2>/dev/null || echo "already cloned"
!cd depth-fm && pip install -q -r requirements.txt
!mkdir -p depth-fm/checkpoints
![ -f depth-fm/checkpoints/depthfm-v1.ckpt ] || wget -q -O depth-fm/checkpoints/depthfm-v1.ckpt https://ommer-lab.com/files/depthfm/depthfm-v1.ckpt

!nvidia-smi

## 4. Select 200 Random Sample Images

In [ ]:
!python depth_comparison/select_samples.py

## 5. Run All 6 Depth Models

Runs sequentially, clears GPU between each model.  
Estimated time: ~30-60 min on T4, ~15-30 min on A100.

To run specific models only, edit the `--models` flag below.

In [ ]:
!python depth_comparison/run_all_models.py
# To run specific models: !python depth_comparison/run_all_models.py --models depth_pro moge2 vggt

## 6. Evaluate & Generate Comparison Grids

In [ ]:
!python depth_comparison/evaluate.py

## 7. View Results

In [ ]:
import json, glob, os
import pandas as pd
from IPython.display import display, Image as IPImage

# Metrics table
with open("depth_comparison/results_summary.json") as f:
    results = json.load(f)

df = pd.DataFrame(results).T
print("=== Results Summary ===")
display(df)

# Pick winner (highest edge_alignment_mean)
scored = {k: v for k, v in results.items() if "edge_alignment_mean" in v}
if scored:
    winner = max(scored, key=lambda k: scored[k]["edge_alignment_mean"])
    print(f"\n🏆 Best edge alignment: {winner} ({scored[winner]['edge_alignment_mean']:.4f})")

# Show comparison grids
grids = sorted(glob.glob("depth_comparison/comparison_grids/*.png"))[:10]
for g in grids:
    print(f"\n{os.path.basename(g)}")
    display(IPImage(filename=g))